
## Part 4 — GenAI: Analyse Follow-up Notes and Identify Reasons for No-Shows

In this part, I used the Gemini API to analyse patient follow-up notes and convert them into structured information.

### Work Completed

1. **Gemini API Integration:** Created an `analyse_note()` function to send patient notes to Gemini and receive structured JSON responses.

2. **Pydantic Validation:** Created a `FollowupInsight` model to validate the root cause, confidence score, summary, and recommended action.

3. **Note Analysis:** Analysed the first 10 follow-up notes. Blank notes were handled separately, and the remaining notes were processed using Gemini.

4. **Save Results:** Stored the analysis, validation status, and original API response in `notes_analysed.csv`.

5. **Error Handling:** Tested invalid responses, including an incorrect confidence score and a missing field, using Pydantic validation.

6. **Prompt Comparison:** Compared a simple V1 prompt with a detailed V2 prompt using notes N007 and N009 to understand how prompt quality affects the results.

7. **Final Analysis:** Merged the analysed notes with the cleaned appointments dataset and created a crosstab to compare root causes with reminder status.

### Outcome

The analysis converts unstructured patient feedback into structured reasons that clinics can use to understand missed appointments and improve their follow-up process.


In [2]:
# Import required libraries
import json
import pandas as pd
!pip install -q google-genai pydantic
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Load follow-up notes
df_notes = pd.read_csv("/content/ecWvPsdV2FNmRZAa.csv")
df_notes.head()

,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


### Task 4.1 — Gemini API Integration
- Connected to the Gemini API.
- Created an `analyse_note()` function to analyse patient follow-up notes.
- Used a structured JSON schema to get four fields: root cause, confidence, summary, and recommended action.
- Tested the function using note N001.

In [ ]:
# Connect Gemini API
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)

In [4]:

# Initialize tracking variables
api_calls = 0
last_raw_response = ""

# Define Gemini analysis function
def analyse_note(note_text):
  # Set system instructions
  SYSTEM_TEXT = """
  You are a AI Assistant which helps the clinics to analyze the follow-up notes after missesed appionments.
  Your job is to analyse the follow-up notes and find the reason behind missed appoinments and  provide a concise structured analysis.
  Return a valid and good infomation which matches the requested JSON schema, and don't add any misinformation in output if you don't have the idea.
   """
   # Create structured analysis prompt
  prompt = f"""
  Context: clinic wants to convert the follow-up notes about missed appoinments into a a good structued output.
  Role: you a an AI assitant.
  Actions: Analyse the follow-up notes and find the reason behind missed appoinments and provide a concise structured analysis.
  The allowed root_cause values are strictly one of these:
- Forgot
- No_Reminder
- Transport
- Distance
- Cost
- Schedule_Conflict
- Health_Issue
- emergency
- illness
- other

Also provide:
* a confidence score between 0 and 1
* a summary of at most 25 words
* a recommended action for the clinic

  Tone: professional, consise and informative
  Format: In json format with the following SCHEMA
  notes: {note_text}
  """
  # Define required JSON structure
  SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}
  # Generate Gemini response
  response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
        max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal")))
  # Track API calls
  global api_calls
  api_calls += 1
   # Store original response
  global last_raw_response
  raw_response = response.text
  last_raw_response = raw_response
  # Convert JSON into dictionary
  result = json.loads(response.text)
  # convert json response into pyhton dictonary
  return result

In [5]:
# N001 note
note_n001 = df_notes[
    df_notes["note_id"] == "N001"
]["note_text"].iloc[0]

# Analyse note
result = analyse_note(note_n001)

# Print the Python dictionary
print(result)

{'root_cause': 'No_Reminder', 'confidence': 0.95, 'summary': 'Patient missed the appointment due to a lack of reminder notification and subsequently forgot the scheduled visit.', 'recommended_action': 'Verify patient contact information and ensure the automated reminder system is functioning correctly for future appointments.'}



### Task 4.2 — Pydantic Validation and Note Analysis
- Created a Pydantic model named `FollowupInsight` to validate the API response.
- Checked whether the root cause belongs to the six allowed categories.
- Analysed the first 10 follow-up notes.
- Handled blank notes without calling the API.
- Added error handling and saved the results in `notes_analysed.csv`, including the raw API response and validation status.


In [6]:
# Define validation model
class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0, le=1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)

In [7]:
# allowed root cases
allowed_root_causes = [
    "Forgot",
    "No_Reminder",
    "Transport",
    "Distance",
    "Cost",
    "Schedule_Conflict",
    "Health_Issue",
    "emergency",
    "illness",
    "other"
]

# validate pydantic
validate = FollowupInsight(**result)


# checking root_case
if validate.root_cause not in allowed_root_causes:
  raise ValueError("Invalid root cause")
else:
  print("Root cause is valid")

Root cause is valid


In [8]:
# Reload follow-up dataset
notes_df = pd.read_csv("/content/ecWvPsdV2FNmRZAa.csv")
notes_df.head()

,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


In [9]:
# Import validation and timing
from pydantic import ValidationError
import time

In [10]:
# Store analysed results
results = []

# Process first ten notes
for index, rows in notes_df.head(10).iterrows():
  # Initialize raw response
  raw_response = ""
   # Extract note details
  note_id = rows["note_id"]
  note_text = rows["note_text"]
   # Handle missing notes
  if pd.isna(note_text) or str(note_text).strip() == "":
      # Create blank note result
      result = {
            "root_cause": "Other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""}
  else:
      try:
          # Calling API
          result = analyse_note(note_text)
          raw_response = last_raw_response
          # Check free-tier rate limit
          time.sleep(13)
            # Validate using Pydantic
          try:
            validate = FollowupInsight(**result)

            if validate.root_cause in allowed_root_causes:
                    status = "valid"
            else:
                    status = "invalid"

          except ValidationError as e:
              print(f"Error in {note_id}: {e}")
              status = "invalid"

            # Keep original response, including missing fields
          result = {
                "root_cause": result.get("root_cause", ""),
                "confidence": result.get("confidence", ""),
                "summary": result.get("summary", ""),
                "recommended_action": result.get("recommended_action", ""),
                "validation_status": status,
                "raw_response": raw_response
            }

      except Exception as e:
           # Handle processing errors
           print(f"API or processing error in {note_id}: {e}")
            # Store invalid result
           result = {
                "root_cause": "",
                "confidence": "",
                "summary": "",
                "recommended_action": "",
                "validation_status": "invalid",
                "raw_response": raw_response
            }

  results.append(result)

In [11]:
# Create results dataframe
results_df = pd.DataFrame(results)
results_df.head()
# adding "note_id" and "appt_id" in results_df
results_df["note_id"] = notes_df.head(10)["note_id"].values
results_df["appointment_id"] = notes_df.head(10)["appointment_id"].values

# final dataframe
results_df = results_df[["note_id", "appointment_id", "root_cause", "confidence", "summary", "recommended_action", "validation_status", "raw_response"]]

# Save analysed notes CSV
results_df.to_csv("notes_analysed.csv", index=False)
print("notes_analysed.csv saved successfully!")
results_df.head(10)

notes_analysed.csv saved successfully!


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,No_Reminder,0.95,Patient missed the appointment due to a lack o...,Verify patient contact information and ensure ...,valid,"{\n ""root_cause"": ""No_Reminder"",\n ""confiden..."
1,N002,A0897,Distance,0.95,Patient missed the appointment due to the clin...,Offer telehealth options for routine check-ups...,valid,"{\n ""root_cause"": ""Distance"",\n ""confidence""..."
2,N003,A0526,Cost,1.00,The patient missed the appointment due to fina...,Provide the patient with information regarding...,valid,"{\n ""root_cause"": ""Cost"",\n ""confidence"": 1...."
3,N004,A0926,Other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness,1.00,Patient missed the appointment due to high fev...,Contact the patient in 3-5 days to check on th...,valid,"{\n ""root_cause"": ""illness"",\n ""confidence"":..."
5,N006,A0507,Schedule_Conflict,1.00,Patient missed the appointment due to an urgen...,Reschedule the appointment to an evening slot ...,valid,"{\n ""root_cause"": ""Schedule_Conflict"",\n ""co..."
6,N007,A0333,No_Reminder,0.95,Patient forgot the appointment due to not rece...,Verify SMS notification system functionality a...,valid,"{\n ""root_cause"": ""No_Reminder"",\n ""confiden..."
7,N008,A0222,Transport,0.95,Patient missed the appointment due to a city b...,Offer telehealth options or assist in coordina...,valid,"{\n ""root_cause"": ""Transport"",\n ""confidence..."
8,N009,A0977,other,0.80,Patient was unreachable via phone after three ...,Send a follow-up letter or email to request a ...,valid,"{\n ""root_cause"": ""other"",\n ""confidence"": 0..."
9,N010,A0596,Schedule_Conflict,0.95,Patient missed the appointment due to a work s...,Update patient profile to prioritize Saturday ...,valid,"{\n ""root_cause"": ""Schedule_Conflict"",\n ""co..."


### Task 4.3 — Validation Testing and Prompt Comparison
- Tested Pydantic using two intentionally incorrect responses:
  - An invalid confidence score.
  - A missing required field.
- Compared a simple V1 prompt with a detailed V2 prompt using notes N007 and N009.
- Checked whether the generated root causes matched the allowed categories.
- Counted the total API calls.

In [12]:
# Test invalid confidence value
# First bad reply
bad_reply_one = {"root_cause": "Forgot","confidence": 1.7,"summary": "Patient forgot about the appoinment","recommended_action": "Send reminder"}
try:
    validate = FollowupInsight(**bad_reply_one)
    print(validate)
except ValidationError as e:
    print("Error in first reply:")
    print(e)

# Second bad reply
bad_reply_two = { "root_cause": "Transport", "confidence": 0.8, "summary": "Patient could not find any transport"}
try:
    validate = FollowupInsight(**bad_reply_two)
    print(validate)
except ValidationError as e:
    print("Error in second reply:")
    print(e)

Error in first reply:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
Error in second reply:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'Transport...not find any transport'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [13]:
# Define response JSON schema
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

# simple prompt
v1_prompt = f"Classify this note"

# Define detailed V2 prompt
v2_prompt = f"""
  Context: clinic wants to convert the follow-up notes about missed appoinments into a a good structued output.
  Role: you a an AI assitant.
  Actions: Analyse the follow-up notes and find the reason behind missed appoinments and provide a concise structured analysis.
  The allowed root causes are:
- Forgot
- No Reminder
- Transport
- Distance
- Cost
- Schedule Conflict
- Health Issue
- other

Also provide:
* a confidence score between 0 and 1
* a summary of at most 25 words
* a recommended action for the clinic

  Tone: professional, consise and informative
  Format: In json format with the following SCHEMA
  """

# Define shared system instructions
SYSTEM_TEXT = """
  You are a AI Assistant which helps the clinics to analyze the follow-up notes after missesed appionments.
  Your job is to analyse the follow-up notes and find the reason behind missed appoinments and  provide a concise structured analysis.
  Return a valid and good infomation which matches the requested JSON schema, and don't add any misinformation in output if you don't have the idea."""

In [14]:
# Define prompt comparison function
def choose_prompt(note_text, prompt):
   # Generate prompt response
  response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=f"{prompt}\n Note :{note_text}",
    config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
        max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal")))
   # Track API calls
  global api_calls
  api_calls += 1
  # Parse generated JSON
  raw_response2 = response.text
  result_2 = json.loads(response.text)
  return result_2

In [15]:
# N007 and N009 note_text
note_n007 = df_notes[
    df_notes["note_id"] == "N007"
]["note_text"].iloc[0]

note_n009 = df_notes[
    df_notes["note_id"] == "N009"
]["note_text"].iloc[0]

# prompt_v1
prompt_v1_n007= choose_prompt(note_n007, v1_prompt)
prompt_v1_n009 = choose_prompt(note_n009, v1_prompt)

# prompt_v2
prompt_v2_n007 = choose_prompt(note_n007, v2_prompt)
prompt_v2_n009 = choose_prompt(note_n009, v2_prompt)

In [16]:
# Store prompt comparison
compare = []

# Collect four test results
test = [
    ("N007", "V1", prompt_v1_n007),
    ("N007", "V2", prompt_v2_n007),
    ("N009", "V1", prompt_v1_n009),
    ("N009", "V2", prompt_v2_n009 )
]

# Validate prompt outputs
for note_id, prompt_no, answer in test:
  root_cause = answer.get("root_cause", "")
  if root_cause in allowed_root_causes:
    status = "valid"
  else:
    status = "invalid"
  compare.append({"note_id": note_id, "prompt_no": prompt_no, "answer": answer, "validation_status": status})

# Display prompt comparison
print(compare)

[{'note_id': 'N007', 'prompt_no': 'V1', 'answer': {'root_cause': 'Forgot appointment due to lack of reminder notification', 'confidence': 0.95, 'summary': 'The patient missed the appointment because they forgot about it and did not receive an SMS reminder. They are requesting to reschedule.', 'recommended_action': "Reschedule the appointment and verify the patient's contact number for SMS notification settings."}, 'validation_status': 'invalid'}, {'note_id': 'N007', 'prompt_no': 'V2', 'answer': {'root_cause': 'Forgot', 'confidence': 0.95, 'summary': 'Patient forgot the appointment and noted they did not receive an SMS reminder. They requested to reschedule.', 'recommended_action': 'Verify patient contact details and ensure the automated SMS reminder system is functioning correctly before rescheduling.'}, 'validation_status': 'valid'}, {'note_id': 'N009', 'prompt_no': 'V1', 'answer': {'root_cause': 'Communication failure', 'confidence': 0.95, 'summary': 'The patient could not be reached

In [17]:
# Display total API calls
print("Total API calls:",api_calls)

Total API calls: 14


### Task 4.4 — Combine Results and Analyse
- Merged `notes_analysed.csv` with `appointments_clean.csv` using `appointment_id`.
- Created a cross-tabulation between `root_cause` and `reminder_sent`.
- Used the results to understand the relationship between missed appointment reasons and reminder status.

In [18]:
import pandas as pd

# Load notes_analysed dataset
notes_df = pd.read_csv("notes_analysed.csv")

# Load appointments_clean dataset
appointments_df = pd.read_csv("appointments_clean.csv")

In [19]:
# Merge both datasets
merge_df = pd.merge(notes_df, appointments_df, on="appointment_id")
merge_df.head()

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,No_Reminder,0.95,Patient missed the appointment due to a lack o...,Verify patient contact information and ensure ...,valid,"{\n ""root_cause"": ""No_Reminder"",\n ""confiden...",P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,Distance,0.95,Patient missed the appointment due to the clin...,Offer telehealth options for routine check-ups...,valid,"{\n ""root_cause"": ""Distance"",\n ""confidence""...",P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,Cost,1.00,The patient missed the appointment due to fina...,Provide the patient with information regarding...,valid,"{\n ""root_cause"": ""Cost"",\n ""confidence"": 1....",P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,Other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,illness,1.00,Patient missed the appointment due to high fev...,Contact the patient in 3-5 days to check on th...,valid,"{\n ""root_cause"": ""illness"",\n ""confidence"":...",P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1


In [20]:
# Display merged row count
print("Merged rows:", len(merge_df))

Merged rows: 10


In [21]:
# Compare causes and reminders
root_cause_table = pd.crosstab( merge_df["root_cause"], merge_df["reminder_sent"])
# Display cross-tabulation
print(root_cause_table)

reminder_sent      No  Yes
root_cause                
Cost                0    1
Distance            1    0
No_Reminder         2    0
Other               1    0
Schedule_Conflict   1    1
Transport           0    1
illness             0    1
other               0    1
